# E15 — Decision-Cost Evaluation (Phase 5)

**Experiment ID:** `E15`. **Spec:** `EXPERIMENT_PLAN.md` §E15 (amended 2026-09-18). **Protocol:** the
2026-09-18 E15 design review (Sidh's decisions D1–D4) and the E15 PRE-REGISTRATION in `DECISIONS.md`,
both written before this notebook read the official test set.

- **Reporting lens (D1).** True high-risk events (final risk ≥ −6, n = 150) are primary; the whole
  official test set is secondary context. **Missed high-risk events is the lead number.**
- **Bound (D2).** Decisions use one-sided upper bounds directly — never the two-sided upper edge.
- **Comparison basis (D3).** Matched alert budget: every method raises the same number of alerts at
  an evaluation point. Cost ratios 5:1, 10:1, 20:1 (missed-high-risk : unnecessary-maneuver).
- **Statistics (D4).** Descriptive only, with event-level bootstrap CIs. No hypothesis test.

> **Caveat carried by every result below.** One-sided upper bounds under-cover on the official test
> set (E11 diagnostic: the one-sided machinery is validated under exchangeability, but rule-derived
> weighting does not restore one-sided validity); persistence's one-sided bound is additionally
> degenerate (a 65.9% zero-atom in its signed scores, Gate 2).

**Scope:** the official test set is read once, for scoring only. Hyperparameters come from the
Phase-2 searches. Results are reported exactly as observed (CLAUDE.md §3, §9); no checkpoint call is
made here.

In [ ]:
# --- Setup + provenance (invariant I4) ------------------------------------------------------
import json, subprocess, sys
from datetime import datetime, timezone

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from kelvins_conformal.config import REPO_ROOT, load_config
from kelvins_conformal.models import decision_runner as DR
from kelvins_conformal.reporting import write_table_atomic

cfg = load_config()
FIGDIR = cfg.path("figures_dir"); FIGDIR.mkdir(parents=True, exist_ok=True)
TABDIR = cfg.path("tables_dir"); TABDIR.mkdir(parents=True, exist_ok=True)

def git_sha():
    try:
        return subprocess.run(["git", "rev-parse", "HEAD"], cwd=str(REPO_ROOT),
                              capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return "UNAVAILABLE"

def save_table(df, name):
    write_table_atomic(df, TABDIR / f"{name}.csv"); print(f"saved: reports/tables/{name}.csv")

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIGDIR / f"{name}.{ext}", dpi=160, bbox_inches="tight", facecolor=fig.get_facecolor())
    print(f"saved: reports/figures/{name}.png|pdf")

PROVENANCE = {"experiment_ids": ["E15"], "git_commit_sha": git_sha(),
              "config_hash": cfg.config_hash, "seeds": list(cfg.train.seeds),
              "cost_ratios": list(cfg.decision_cost.cost_ratios),
              "budget_fractions": list(cfg.decision_cost.budget_fractions),
              "primary_budget": cfg.decision_cost.primary_budget,
              "bootstrap_resamples": cfg.bootstrap.n_resamples,
              "executed_utc": datetime.now(timezone.utc).isoformat(), "python": sys.version.split()[0]}
print(json.dumps(PROVENANCE, indent=2))

# Chart chrome + categorical slots 1-6 of the dataviz reference palette (light mode), in fixed
# order and validated with its palette script before use. Colour follows the scoring rule, never
# its rank; text stays in ink colours.
SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
SERIES = [
    ("point", "persistence", "persistence — point prediction", "#2a78d6"),
    ("point", "gbm", "GBM — point prediction", "#eb6834"),
    ("point", "gru", "GRU — point prediction", "#1baf7a"),
    ("point", "mc_dropout", "MC-dropout — point prediction (MC mean)", "#eda100"),
    ("E12_cqr_weighted_rule_upper", "gbm", "GBM — one-sided CQR bound (rule-weighted)", "#e87ba4"),
    ("E8_bayes_upper", "mc_dropout", "MC-dropout — Bayesian one-sided bound (E8)", "#008300"),
]
plt.rcParams.update({"axes.edgecolor": AXIS, "axes.labelcolor": INK2, "xtick.color": MUTED,
                     "ytick.color": MUTED, "text.color": INK, "axes.titlecolor": INK})

## 1. Run E15

In [ ]:
RES = DR.run_e15(cfg)
meta = RES["meta"]
print(json.dumps({k: meta[k] for k in ("seeds", "levels", "primary_level", "primary_budget",
                                       "cost_ratios", "n_boot", "n_test", "n_high_risk",
                                       "seed_seconds")}, indent=2, default=str))
for key in ("decisions", "bound_coverage", "cqr_selftest", "paired_differences", "alert_identity",
            "budgets", "positivity", "search_integrity", "tradeoff_curves"):
    save_table(RES[key], f"e15_{key}")
PRIM, PB, CAVEAT = meta["primary_level"], meta["primary_budget"], meta["caveat"]
BUDGETS = RES["budgets"]
K_PRIMARY = int(BUDGETS.loc[BUDGETS["primary"], "K"].iloc[0])
dec = RES["decisions"]
display(BUDGETS)

## 2. Integrity checks, read first

- **Search integrity (pre-registration §8).** Adding the `decision_cost` config block changed the
  config hash, so the GBM, GRU and MC-dropout searches re-ran. They are seeded, so each must
  reproduce every earlier cache.
- **Alert identity (§4(i)).** At a matched budget, a one-sided split or weighted conformal bound is
  expected to raise exactly the same alerts as its own point prediction. One-sided CQR is expected to
  raise the same alerts as its GBM upper quantile head. Expected difference: 0, except where float
  rounding creates a new tie.
- **Positivity.** Supported-region counts (Q-SEL-03).

In [ ]:
si = RES["search_integrity"]
display(si)
compared = si[si["status"] == "compared"]
SEARCH_OK = bool(len(compared)) and bool(compared["best_params_equal"].all())
print(f"All re-run searches reproduce the earlier best_params: {SEARCH_OK}")

ident = RES["alert_identity"]
display(ident.groupby(["method", "learner", "reference"])["max_abs_alert_difference"].max().reset_index())
IDENT_MAX = float(ident["max_abs_alert_difference"].max())
print(f"Largest alert difference vs the pre-registered reference, all levels and budgets: {IDENT_MAX:.3g}")
display(RES["positivity"])

## 3. One-sided CQR machinery validation on the exchangeable self-split (pre-registration §5)

E12 computed two-sided CQR only, so the one-sided CQR bound is new at E15. It is trusted only if its
coverage on the exchangeable self-test split tracks nominal (the Clopper–Pearson interval contains
nominal).

In [ ]:
st = RES["cqr_selftest"].copy()
st["cp_contains_nominal"] = (st["cp_lo"] <= st["nominal"]) & (st["nominal"] <= st["cp_hi"])
display(st.round(4))
CQR_VALIDATED = bool(st["cp_contains_nominal"].all())
print(f"One-sided CQR validated on the exchangeable self-split at every level: {CQR_VALIDATED}")

## 4. PRIMARY — high-risk events, prevalence-matched budget, nominal 90% (D1)

**Missed high-risk events is the lead number.** Every method raises the same number of alerts K.
Values are means over 3 seeds; brackets are seed-averaged 95% event-level bootstrap intervals.

In [ ]:
def ci(row, name, digits=1, scale=1.0, signed=False):
    f = "+.{0}f".format(digits) if signed else ".{0}f".format(digits)
    return (f"{row[name] * scale:{f}} [{row[name + '_lo'] * scale:{f}}, "
            f"{row[name + '_hi'] * scale:{f}}]")

def rows_at(level, budget):
    order = {m: i for i, m in enumerate(DR.METHODS)}
    v = dec[(dec["nominal"] == level) & (dec["budget"] == budget)].copy()
    v["_order"] = v["method"].map(order)
    return v.sort_values(["missed_high_risk", "_order", "learner"]).drop(columns="_order")

prim = rows_at(PRIM, PB)
table_hr = pd.DataFrame({
    "method": prim["method"], "learner": prim["learner"], "K (alerts)": prim["K"],
    "missed high-risk events [95% CI]": [ci(r, "missed_high_risk") for _, r in prim.iterrows()],
    "miss rate % [95% CI]": [ci(r, "miss_rate_high_risk", scale=100) for _, r in prim.iterrows()],
    "recall % [95% CI]": [ci(r, "recall_high_risk", scale=100) for _, r in prim.iterrows()],
    "sd of missed across seeds": prim["missed_high_risk_sd_across_seeds"].round(2),
}).reset_index(drop=True)
display(table_hr)
save_table(table_hr.assign(caveat=CAVEAT), "e15_primary_high_risk")
print(f"n_high_risk = {meta['n_high_risk']}, K = {K_PRIMARY}.")
print("CAVEAT:", CAVEAT)

## 5. SECONDARY — whole population, same budget and level (D1)

Unnecessary maneuvers, false-positive rate, precision, F2 and cost all count non-high-risk events by
definition, so they cannot be computed on the high-risk events alone (pre-registration §3). At a
matched budget, cost is C_r = (r + 1)·FN + K − n_HR (§4(ii)), so the cost columns order methods
exactly as missed high-risk events does.

In [ ]:
RATIOS = meta["cost_ratios"]
cost_cols = {f"cost {r:g}:1 [95% CI]": [ci(r_, DR.dc.cost_key(r), digits=0) for _, r_ in prim.iterrows()]
             for r in RATIOS}
table_pop = pd.DataFrame({
    "method": prim["method"], "learner": prim["learner"],
    "unnecessary maneuvers [95% CI]": [ci(r, "unnecessary_maneuvers") for _, r in prim.iterrows()],
    "false-positive rate % [95% CI]": [ci(r, "false_positive_rate", digits=2, scale=100) for _, r in prim.iterrows()],
    "precision % [95% CI]": [ci(r, "precision", scale=100) for _, r in prim.iterrows()],
    "F2 [95% CI]": [ci(r, "f2", digits=3) for _, r in prim.iterrows()],
    **cost_cols,
}).reset_index(drop=True)
display(table_pop)
save_table(table_pop.assign(caveat=CAVEAT), "e15_secondary_whole_population")

orderings = {name: tuple(prim.sort_values([name, "method", "learner"])[["method", "learner"]].itertuples(index=False))
             for name in ["missed_high_risk", *[DR.dc.cost_key(r) for r in RATIOS]]}
COST_ORDER_IDENTICAL = len(set(orderings.values())) == 1
print(f"Method ordering identical across missed high-risk events and all three cost ratios: {COST_ORDER_IDENTICAL}")
print("CAVEAT:", CAVEAT)

## 6. Paired differences in missed high-risk events (pre-registration §6)

Each calibrated method is compared with (a) its own learner's point prediction and (b) the E8
Bayesian bound, on the same bootstrap resamples, at the primary budget and level. **Positive = the
method misses MORE high-risk events than the comparator.** Descriptive (D4): no p-values.

In [ ]:
pdiff = RES["paired_differences"].copy()
pdiff["difference in missed high-risk events [95% CI]"] = [
    f"{r.diff_missed_high_risk:+.1f} [{r.diff_lo:+.1f}, {r.diff_hi:+.1f}]" for r in pdiff.itertuples()]
pdiff["interval position"] = np.select(
    [pdiff["diff_hi"] < 0, pdiff["diff_lo"] > 0], ["entirely below 0", "entirely above 0"], "includes 0")
view = pdiff[["method", "learner", "vs", "difference in missed high-risk events [95% CI]", "interval position"]]
display(view)
save_table(view.assign(caveat=CAVEAT), "e15_paired_differences_table")
print("CAVEAT:", CAVEAT)

## 7. Tradeoff curves — missed high-risk events vs. unnecessary maneuvers (spec figure)

Swept over every alert budget K at nominal 90%. Dots mark the prevalence-matched budget. The E10/E11
one-sided conformal bounds are not drawn separately because of the alert-identity result in §2 (the
caption states what was verified).

In [ ]:
cur = RES["tradeoff_curves"]
zoom_k = int(BUDGETS["K"].max())
fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.0), facecolor=SURFACE)
for ax, kmax, title in ((axes[0], meta["n_test"], "Full budget sweep, K = 1 … N"),
                        (axes[1], zoom_k, f"Operational range, K ≤ {zoom_k} (the largest pre-registered budget)")):
    ax.set_facecolor(SURFACE)
    for method, learner, label, colour in SERIES:
        g = cur[(cur["method"] == method) & (cur["learner"] == learner) & (cur["K"] <= kmax)]
        ax.plot(g["unnecessary_maneuvers"], g["missed_high_risk"], color=colour, lw=1.6, label=label)
        p = g[g["K"] == K_PRIMARY]
        ax.plot(p["unnecessary_maneuvers"], p["missed_high_risk"], "o", ms=6.5, color=colour,
                mec=SURFACE, mew=1.5)
    ax.grid(True, color=GRID, lw=0.6)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.set_xlabel("unnecessary maneuvers (whole population)")
    ax.set_ylabel(f"missed high-risk events (of {meta['n_high_risk']})")
    ax.set_title(title, fontsize=10, loc="left", color=INK)
axes[1].legend(frameon=False, fontsize=8, loc="upper right")
fig.suptitle(f"E15: missed high-risk events vs. unnecessary maneuvers at matched alert budgets, nominal "
             f"{PRIM:.0%} (dots: prevalence-matched budget, K = {K_PRIMARY})", x=0.01, ha="left", fontsize=11)
identity_note = ("E10 and E11 one-sided conformal bounds are not drawn: at every budget they raise exactly the "
                 "same alerts as their own point prediction (pre-registration §4(i), verified in §2)."
                 if IDENT_MAX == 0.0 else
                 f"E10 and E11 one-sided conformal bounds are not drawn; their largest alert difference from their "
                 f"own point prediction is {IDENT_MAX:.3g} (see §2).")
fig.text(0.01, -0.02, identity_note + "\nCaveat: " + CAVEAT, fontsize=7.5, color=INK2, ha="left", va="top",
         wrap=True)
fig.tight_layout(rect=(0, 0.02, 1, 0.95))
save_fig(fig, "e15_tradeoff_missed_vs_unnecessary")
plt.show()

## 8. Decision-cost summary across budgets, levels and cost ratios (spec table)

The full table, every metric with its interval, is saved as `e15_decision_cost_summary.csv`. Shown
here: missed high-risk events at nominal 90% for every pre-registered budget.

In [ ]:
summary = dec.sort_values(["nominal", "K", "method", "learner"])
save_table(summary.assign(caveat=CAVEAT), "e15_decision_cost_summary")
pivot = (dec[dec["nominal"] == PRIM]
         .pivot_table(index=["method", "learner"], columns="budget", values="missed_high_risk")
         [list(BUDGETS["budget"])].round(1))
pivot.columns = [f"{b} (K={k})" for b, k in zip(BUDGETS["budget"], BUDGETS["K"])]
display(pivot)
print("CAVEAT:", CAVEAT)

## 9. The D2 caveat, quantified — realised one-sided coverage of each bound

These are one-sided coverage figures (y ≤ bound) on the official test set, for the whole supported
population and for high-risk events. Persistence's bound carries its separate degeneracy caveat
(Gate 2).

In [ ]:
bc = RES["bound_coverage"]
v = bc[bc["nominal"] == PRIM].copy()
v["coverage [CP 95% CI]"] = [f"{r.coverage:.3f} [{r.cp_lo:.3f}, {r.cp_hi:.3f}]" for r in v.itertuples()]
cov_view = v.pivot_table(index=["method", "learner"], columns="population", values="coverage [CP 95% CI]",
                         aggfunc="first")
display(cov_view)
save_table(bc.assign(caveat=CAVEAT), "e15_bound_coverage_table")
print(f"Nominal one-sided level: {PRIM:.0%}.  CAVEAT:", CAVEAT)

## 10. E15 findings — measurement only, reported exactly as observed

In [ ]:
def pick(method, learner, level=None, budget=None):
    level = PRIM if level is None else level
    budget = PB if budget is None else budget
    return dec[(dec["method"] == method) & (dec["learner"] == learner) & (dec["nominal"] == level)
               & (dec["budget"] == budget)].iloc[0]

lines = []
for method, learner, label, _ in SERIES:
    r = pick(method, learner)
    lines.append(f"    {label:<46s} missed {ci(r, 'missed_high_risk')}  | unnecessary {ci(r, 'unnecessary_maneuvers')}")
pair_lines = [f"    {r.method} / {r.learner} vs {r.vs}: {r.diff_missed_high_risk:+.1f} "
              f"[{r.diff_lo:+.1f}, {r.diff_hi:+.1f}] ({p})"
              for r, p in zip(pdiff.itertuples(), pdiff["interval position"])]
print(f"""
E15 / DECISION COST — WHAT THE BATCH SHOWS (measurement only, exactly as observed)

 SCOPE: official test N = {meta['n_test']}, true high-risk n = {meta['n_high_risk']}; prevalence-matched
 budget K = {K_PRIMARY}; nominal {PRIM:.0%}; seeds {meta['seeds']}; {meta['n_boot']} event-level resamples.

 INTEGRITY: searches reproduce earlier caches = {SEARCH_OK}; largest alert difference vs the
 pre-registered reference = {IDENT_MAX:.3g}; one-sided CQR validated on the self-split = {CQR_VALIDATED};
 method ordering identical across missed high-risk events and all cost ratios = {COST_ORDER_IDENTICAL}.

 PRIMARY — missed high-risk events at K = {K_PRIMARY} (mean over seeds [95% CI]):
{chr(10).join(lines)}

 PAIRED DIFFERENCES in missed high-risk events (positive = misses more; descriptive, D4):
{chr(10).join(pair_lines)}

 CAVEAT: {CAVEAT}

 NOT DECIDED HERE (CLAUDE.md §3, §9, §13.7): whether the E15 failure or success criterion is met as a
 judgement; whether the pre-registration (including its §3 lens reading and §4 identities) is
 confirmed; any framing consequence; and anything in E16-E18. Execution stops at this checkpoint.
""")
(cfg.path("reports_dir") / "05_decision_cost_provenance.json").write_text(json.dumps(PROVENANCE, indent=2), encoding="utf-8")
print("provenance:", cfg.path("reports_dir") / "05_decision_cost_provenance.json")